# Dimensionality Reduction: PCA & ICA

## 0. Step-by-Step Worked Example — Start Here (Beginner Friendly)

> 🧑‍🎓 **New to this topic? Start here.** This is a gentle, fully runnable walkthrough that
> builds up *every* idea in this lesson one tiny step at a time. Each step **prints** the
> numbers it computes and **draws a picture** so you can *see* what is happening. Run the
> cells in order from top to bottom. Nothing here needs the internet or any downloaded data.

### The Big Picture — What You'll Learn

In plain terms, here is what the steps below will show you:

- **PCA preprocessing** centers the data, optionally standardizes scales, and builds a covariance matrix.
- **PCA eigenvectors** give orthogonal directions ordered by variance, with eigenvalues as variance captured.
- **PCA projection** compresses points into PC coordinates and reconstructs them with measurable error.
- **ICA** whitens mixed signals and then searches for a non-Gaussian rotation that separates independent sources.

Everything below (starting at **§1 Overview**) develops these same ideas with fuller derivations,
worked examples, and an interactive reconstruction experiment.

**What we will build, step by step:**
1. **PCA centering, scaling, and covariance** — prepare the data cloud and measure feature spread.
2. **PCA eigenvectors as principal directions** — find the axes that capture the most variance.
3. **PCA projection, reconstruction, and explained variance** — compress points and measure what was kept.
4. **ICA source separation** — unmix blended non-Gaussian signals into independent-looking sources.

### Step 0 — Set up our tools

We import NumPy (arrays + linear algebra) and Matplotlib (pictures). We fix a random **seed**
so every run gives the same numbers, then define a tiny `log()` helper so printed output is
self-explanatory.

In [ ]:
import numpy as np                       # NumPy: arrays, centering, covariance, eigenvectors, and signal mixing.
import matplotlib.pyplot as plt          # Matplotlib: draw PCA geometry and ICA signal separation.

np.random.seed(0)                         # Fix the seed so every run prints the SAME numbers.
plt.rcParams["figure.figsize"] = (7, 4)   # Use a comfortable default plot size.


def log(label, value):                    # A tiny logger so each printed line explains itself.
    print(f"[{label}] {value}")           # Format is: [what this is] the value.

log("setup", "tools ready — NumPy + Matplotlib imported, seed fixed to 0")

▶ What you'll see: one line confirming the tools are ready.

### Step 1 — PCA starts with centering, optional scaling, and covariance

PCA studies **spread around the data's own center**, so we subtract each feature mean before
measuring variance. If feature scales are very different, standardizing also prevents a large-unit
feature from dominating the covariance matrix.

In [ ]:
X_pca_demo = np.array([[1.0, 1.1], [2.0, 1.5], [2.8, 2.1], [3.6, 2.6], [4.5, 3.2], [5.1, 3.7]])  # Tiny tilted 2-D dataset.
mean_pca_demo = X_pca_demo.mean(axis=0)  # Compute one mean per feature column.
X_centered_demo = X_pca_demo - mean_pca_demo  # Center the data so PCA measures shape, not location.
std_pca_demo = X_centered_demo.std(axis=0)  # Compute feature standard deviations for optional scaling.
X_scaled_demo = X_centered_demo / std_pca_demo  # Standardize columns to unit scale when units are incomparable.
m_pca_demo = X_centered_demo.shape[0]  # Count examples for the CS 229 covariance normalization.
cov_pca_demo = (X_centered_demo.T @ X_centered_demo) / m_pca_demo  # Compute Sigma = X_centered^T X_centered / m.
cov_scaled_demo = (X_scaled_demo.T @ X_scaled_demo) / m_pca_demo  # Compute covariance after standardizing scales.

log("feature means", np.round(mean_pca_demo, 3))  # Print the center being subtracted.
log("centered column means", np.round(X_centered_demo.mean(axis=0), 10))  # Verify centering worked.
log("feature stds", np.round(std_pca_demo, 3))  # Print scales before optional standardization.
log("covariance matrix", np.round(cov_pca_demo, 3))  # Print variances and covariance on centered data.
log("scaled covariance", np.round(cov_scaled_demo, 3))  # Print the standardized covariance for comparison.

fig_center_demo, axes_center_demo = plt.subplots(1, 2, figsize=(10, 3.8))  # Create raw and centered panels.
axes_center_demo[0].scatter(X_pca_demo[:, 0], X_pca_demo[:, 1], s=90, color="slateblue")  # Draw original points.
axes_center_demo[0].scatter(mean_pca_demo[0], mean_pca_demo[1], s=130, color="black", marker="X", label="mean")  # Mark the data mean.
axes_center_demo[0].set_title("raw data with mean")  # Title the raw panel.
axes_center_demo[0].set_xlabel("feature 1")  # Label the first feature.
axes_center_demo[0].set_ylabel("feature 2")  # Label the second feature.
axes_center_demo[0].legend()  # Identify the mean marker.
axes_center_demo[1].scatter(X_centered_demo[:, 0], X_centered_demo[:, 1], s=90, color="seagreen")  # Draw centered points.
axes_center_demo[1].axhline(0, color="gray", linestyle="--")  # Draw centered horizontal axis.
axes_center_demo[1].axvline(0, color="gray", linestyle="--")  # Draw centered vertical axis.
axes_center_demo[1].set_title("centered cloud")  # Title the centered panel.
axes_center_demo[1].set_xlabel("centered feature 1")  # Label centered feature 1.
axes_center_demo[1].set_ylabel("centered feature 2")  # Label centered feature 2.
plt.tight_layout()  # Keep panel labels readable.
plt.show()  # Render the centering visualization.

▶ What you'll see: the same tilted cloud moves to mean zero, and the covariance matrix records its diagonal spread and off-diagonal co-movement.

### Step 2 — PCA eigenvectors: principal directions are variance-maximizing axes

For any unit direction $u$, projected variance is $u^T\Sigma u$. The directions that maximize
this quantity are eigenvectors of the covariance matrix, and their eigenvalues tell us how much
variance each principal component captures.

In [ ]:
eigvals_pca_demo, eigvecs_pca_demo = np.linalg.eigh(cov_pca_demo)  # Eigen-decompose the symmetric covariance matrix.
order_pca_demo = np.argsort(eigvals_pca_demo)[::-1]  # Sort component indices from largest variance to smallest.
eigvals_pca_demo = eigvals_pca_demo[order_pca_demo]  # Reorder eigenvalues by explained variance.
eigvecs_pca_demo = eigvecs_pca_demo[:, order_pca_demo]  # Reorder eigenvectors to match the eigenvalues.
ratio_pca_demo = eigvals_pca_demo / eigvals_pca_demo.sum()  # Convert eigenvalues into explained-variance ratios.
unit_diag_demo = np.array([1.0, 1.0]) / np.sqrt(2.0)  # A hand-picked comparison direction.
var_diag_demo = unit_diag_demo.T @ cov_pca_demo @ unit_diag_demo  # Projected variance along the comparison direction.
var_pc1_demo = eigvecs_pca_demo[:, 0].T @ cov_pca_demo @ eigvecs_pca_demo[:, 0]  # Projected variance along PC1.
var_pc2_demo = eigvecs_pca_demo[:, 1].T @ cov_pca_demo @ eigvecs_pca_demo[:, 1]  # Projected variance along PC2.

log("eigenvalues", np.round(eigvals_pca_demo, 3))  # Print variance captured by each PC.
log("eigenvectors columns", np.round(eigvecs_pca_demo, 3))  # Print principal axes as columns.
log("variance along PC1", round(float(var_pc1_demo), 3))  # Show PC1 variance equals the largest eigenvalue.
log("variance along PC2", round(float(var_pc2_demo), 3))  # Show PC2 variance equals the second eigenvalue.
log("variance along diagonal", round(float(var_diag_demo), 3))  # Compare with an arbitrary unit direction.
log("explained variance ratios", np.round(ratio_pca_demo, 3))  # Print variance fractions.

plt.scatter(X_centered_demo[:, 0], X_centered_demo[:, 1], s=90, alpha=0.7, color="slateblue")  # Draw centered points under the arrows.
for comp_idx_demo in range(2):  # Draw both principal-component directions.
    arrow_demo = eigvecs_pca_demo[:, comp_idx_demo] * np.sqrt(eigvals_pca_demo[comp_idx_demo])  # Scale unit vector by standard deviation.
    color_demo = ["crimson", "seagreen"][comp_idx_demo]  # Choose a stable color per component.
    plt.arrow(0, 0, arrow_demo[0], arrow_demo[1], width=0.035, color=color_demo, length_includes_head=True)  # Draw one PC arrow.
    plt.text(arrow_demo[0] * 1.12, arrow_demo[1] * 1.12, f"PC{comp_idx_demo + 1}", color=color_demo)  # Label the arrow tip.
plt.axhline(0, color="gray", linestyle="--")  # Draw the centered horizontal axis.
plt.axvline(0, color="gray", linestyle="--")  # Draw the centered vertical axis.
plt.xlabel("centered feature 1")  # Label the centered x-axis.
plt.ylabel("centered feature 2")  # Label the centered y-axis.
plt.title("PCA eigenvectors point along maximum variance")  # Title the principal-axis plot.
plt.axis("equal")  # Preserve geometric angles.
plt.show()  # Render the PCA direction visualization.

▶ What you'll see: PC1 points along the longest direction of the cloud, and its eigenvalue/ratio is much larger than PC2's.

### Step 3 — PCA projection and reconstruction: compress, then measure loss

Once the principal directions are known, reducing dimension is a dot product: $Z=\widetilde XU_k$.
Reconstruction maps back with $\widehat X=ZU_k^T+\mu$, but any discarded perpendicular component
becomes reconstruction error.

In [ ]:
U1_pca_demo = eigvecs_pca_demo[:, :1]  # Keep only the top principal direction for 1-D compression.
Z_pca_demo = X_centered_demo @ U1_pca_demo  # Project each centered point into one PCA coordinate.
X_recon_centered_demo = Z_pca_demo @ U1_pca_demo.T  # Reconstruct centered points from the one kept component.
X_recon_demo = X_recon_centered_demo + mean_pca_demo  # Add the mean back to return to original feature space.
recon_error_demo = np.mean(np.sum((X_pca_demo - X_recon_demo) ** 2, axis=1))  # Average squared reconstruction error per point.
cumulative_ratio_demo = np.cumsum(ratio_pca_demo)  # Compute cumulative explained variance by component count.

log("compressed shape", Z_pca_demo.shape)  # Print that 2-D rows became 1-D coordinates.
log("first three PCA coordinates", np.round(Z_pca_demo[:3, 0], 3))  # Print example compressed values.
log("first three reconstructions", np.round(X_recon_demo[:3], 3))  # Print approximate original-space points.
log("average reconstruction error", round(float(recon_error_demo), 4))  # Print the price of keeping only PC1.
log("cumulative variance", np.round(cumulative_ratio_demo, 3))  # Print how much variance 1 or 2 PCs keep.

plt.scatter(X_centered_demo[:, 0], X_centered_demo[:, 1], s=90, color="lightgray", edgecolor="black", label="centered data")  # Draw original centered points.
plt.scatter(X_recon_centered_demo[:, 0], X_recon_centered_demo[:, 1], s=90, color="crimson", label="1-D reconstructions")  # Draw projected/reconstructed points.
for original_demo, projected_demo in zip(X_centered_demo, X_recon_centered_demo):  # Connect each point to its projection.
    plt.plot([original_demo[0], projected_demo[0]], [original_demo[1], projected_demo[1]], color="gray", linestyle="--", linewidth=1)  # Show discarded residual.
line_span_demo = np.array([-3.5, 3.5])[:, None] * eigvecs_pca_demo[:, 0][None, :]  # Build endpoints for the PC1 line.
plt.plot(line_span_demo[:, 0], line_span_demo[:, 1], color="crimson", linewidth=2, label="PC1 axis")  # Draw the kept one-dimensional subspace.
plt.xlabel("centered feature 1")  # Label the centered x-axis.
plt.ylabel("centered feature 2")  # Label the centered y-axis.
plt.title("Projection drops points onto the PC1 line")  # Title the projection/reconstruction picture.
plt.legend()  # Identify data, projections, and axis.
plt.axis("equal")  # Preserve geometric projection angles.
plt.show()  # Render the PCA compression visualization.

▶ What you'll see: every point drops onto the red PC1 line; the dashed segments are the information lost by one-dimensional compression.

### Step 4 — ICA source separation: unmix non-Gaussian signals

ICA assumes observations are mixtures $x=As$ of hidden independent sources. PCA can whiten the
mixtures so they are uncorrelated, but ICA goes further by rotating whitened data toward components
that look strongly non-Gaussian and therefore more source-like.

In [ ]:
time_ica_demo = np.linspace(0.0, 1.0, 160)  # Short time axis for two hidden signals.
source1_demo = np.sign(np.sin(2 * np.pi * 3 * time_ica_demo))  # Non-Gaussian square-wave source.
source2_demo = 2.0 * ((5 * time_ica_demo) % 1.0) - 1.0  # Non-Gaussian sawtooth-like source.
S_ica_demo = np.column_stack([source1_demo, source2_demo])  # Store hidden sources as columns.
S_ica_demo = (S_ica_demo - S_ica_demo.mean(axis=0)) / S_ica_demo.std(axis=0)  # Standardize source scales.
A_ica_demo = np.array([[1.0, 0.7], [0.45, 1.0]])  # Mixing matrix that blends both sources into both sensors.
X_mix_demo = S_ica_demo @ A_ica_demo.T  # Observed mixtures, row-wise version of x = A s.
X_mix_centered_demo = X_mix_demo - X_mix_demo.mean(axis=0)  # Center mixtures before whitening.
cov_mix_demo = (X_mix_centered_demo.T @ X_mix_centered_demo) / (X_mix_centered_demo.shape[0] - 1)  # Mixed-sensor covariance.
vals_mix_demo, vecs_mix_demo = np.linalg.eigh(cov_mix_demo)  # Eigen-decompose covariance for whitening.
white_mix_demo = vecs_mix_demo @ np.diag(1.0 / np.sqrt(vals_mix_demo)) @ vecs_mix_demo.T  # Whitening matrix makes covariance near identity.
X_white_demo = X_mix_centered_demo @ white_mix_demo  # Whitened mixtures are uncorrelated but still rotated.
angles_ica_demo = np.linspace(0.0, np.pi, 181)  # Try many possible rotations after whitening.
scores_ica_demo = []  # Store a non-Gaussianity score for each rotation.

for angle_ica_demo in angles_ica_demo:  # Scan rotations because whitening leaves a rotational ambiguity.
    R_ica_demo = np.array([[np.cos(angle_ica_demo), -np.sin(angle_ica_demo)], [np.sin(angle_ica_demo), np.cos(angle_ica_demo)]])  # 2-D rotation matrix.
    Y_try_demo = X_white_demo @ R_ica_demo  # Candidate recovered components.
    kurt_try_demo = np.mean(Y_try_demo**4, axis=0) - 3.0  # Excess-kurtosis-like non-Gaussianity per component.
    scores_ica_demo.append(np.sum(np.abs(kurt_try_demo)))  # Prefer rotations with strongly non-Gaussian marginals.

best_idx_ica_demo = int(np.argmax(scores_ica_demo))  # Choose the rotation with the largest non-Gaussianity score.
best_angle_ica_demo = angles_ica_demo[best_idx_ica_demo]  # Store the winning angle.
R_best_ica_demo = np.array([[np.cos(best_angle_ica_demo), -np.sin(best_angle_ica_demo)], [np.sin(best_angle_ica_demo), np.cos(best_angle_ica_demo)]])  # Rebuild best rotation.
Y_ica_demo = X_white_demo @ R_best_ica_demo  # ICA-style recovered components.
corr_ica_demo = np.abs(np.corrcoef(Y_ica_demo.T, S_ica_demo.T)[:2, 2:])  # Compare to true sources for teaching only; sign/order can flip.

log("mixing matrix A", A_ica_demo)  # Print how sensors blend sources.
log("mixed covariance", np.round(cov_mix_demo, 3))  # Print correlation before whitening.
log("whitened covariance", np.round(np.cov(X_white_demo, rowvar=False), 3))  # Print near-identity covariance after whitening.
log("best ICA rotation angle", round(float(best_angle_ica_demo), 3))  # Print the selected source-seeking rotation.
log("abs corr recovered vs true", np.round(corr_ica_demo, 3))  # Print recovery quality up to sign and order.

fig_ica_demo, axes_ica_demo = plt.subplots(3, 1, figsize=(8, 6), sharex=True)  # Create stacked source/mixture/recovery panels.
axes_ica_demo[0].plot(time_ica_demo, S_ica_demo[:, 0], label="source 1")  # Draw hidden source 1.
axes_ica_demo[0].plot(time_ica_demo, S_ica_demo[:, 1], label="source 2", alpha=0.8)  # Draw hidden source 2.
axes_ica_demo[0].set_title("hidden independent sources")  # Title the source panel.
axes_ica_demo[0].legend(loc="upper right")  # Label sources.
axes_ica_demo[1].plot(time_ica_demo, X_mix_demo[:, 0], label="mixed sensor 1")  # Draw observed mixture 1.
axes_ica_demo[1].plot(time_ica_demo, X_mix_demo[:, 1], label="mixed sensor 2", alpha=0.8)  # Draw observed mixture 2.
axes_ica_demo[1].set_title("observed mixtures")  # Title the mixture panel.
axes_ica_demo[1].legend(loc="upper right")  # Label mixtures.
axes_ica_demo[2].plot(time_ica_demo, Y_ica_demo[:, 0], label="recovered 1")  # Draw recovered component 1.
axes_ica_demo[2].plot(time_ica_demo, Y_ica_demo[:, 1], label="recovered 2", alpha=0.8)  # Draw recovered component 2.
axes_ica_demo[2].set_title("ICA-style recovered components")  # Title the recovery panel.
axes_ica_demo[2].set_xlabel("time")  # Label the shared time axis.
axes_ica_demo[2].legend(loc="upper right")  # Label recovered components.
plt.tight_layout()  # Keep stacked panels readable.
plt.show()  # Render the ICA source-separation visualization.

▶ What you'll see: the observed sensors are blended, while the recovered components line up strongly with the original sources up to sign and order.

---

## 1. Overview

Dimensionality reduction replaces many original features with a smaller number of learned coordinates. The goal is not merely to delete columns; it is to preserve the structure that matters for visualization, compression, denoising, or downstream modeling.

Principal component analysis (PCA) finds orthogonal directions of maximum variance. Independent component analysis (ICA) instead asks whether the observed variables are mixtures of hidden statistically independent sources.

**One-line intuition:** PCA rotates the coordinate system toward the widest directions of the data cloud; ICA tries to unmix blended signals into independent causes.

## 2. Key Idea

### PCA as centering, covariance, eigenvectors, and projection

Suppose the dataset has $m$ examples and $n$ features. Store row $i$ as $x^{(i)T}$ and the full data matrix as

$$
X=\begin{bmatrix}
---(x^{(1)})^T---\\
---(x^{(2)})^T---\\
\vdots\\
---(x^{(m)})^T---
\end{bmatrix}\in\mathbb{R}^{m\times n}.
$$

For PCA, first center each feature:

$$
\mu_j=\frac{1}{m}\sum_{i=1}^{m}x_j^{(i)},
\qquad
\widetilde{x}_j^{(i)}=x_j^{(i)}-\mu_j.
$$

If features are measured on very different scales, normalize them too:

$$
x_j^{(i)}\leftarrow \frac{x_j^{(i)}-\mu_j}{\sigma_j},
\qquad
\sigma_j^2=\frac{1}{m}\sum_{i=1}^{m}(x_j^{(i)}-\mu_j)^2.
$$

In matrix form, let $\widetilde{X}$ be the centered or standardized matrix. The CS 229 PCA covariance matrix is

$$
\Sigma=\frac{1}{m}\sum_{i=1}^{m}\widetilde{x}^{(i)}\widetilde{x}^{(i)T}
       =\frac{1}{m}\widetilde{X}^T\widetilde{X}
       \in\mathbb{R}^{n\times n}.
$$

Because $\Sigma$ is symmetric, the spectral theorem gives an orthonormal eigenbasis:

$$
\Sigma=U\Lambda U^T,
\qquad
\Lambda=\operatorname{diag}(\lambda_1,\ldots,\lambda_n),
\qquad
\lambda_1\ge\lambda_2\ge\cdots\ge\lambda_n\ge 0.
$$

Each eigenvector $u_j$ solves

$$
\Sigma u_j=\lambda_j u_j.
$$

The direction $u_1$ maximizes variance among all unit directions. To see this, the variance of the scalar projection onto a unit vector $u$ is

$$
\operatorname{Var}(\widetilde{x}^Tu)=\frac{1}{m}\sum_{i=1}^{m}(\widetilde{x}^{(i)T}u)^2
=u^T\left(\frac{1}{m}\widetilde{X}^T\widetilde{X}\right)u
=u^T\Sigma u.
$$

Maximizing $u^T\Sigma u$ subject to $u^Tu=1$ gives the Lagrangian

$$
\mathcal{L}(u,\lambda)=u^T\Sigma u-\lambda(u^Tu-1).
$$

Differentiate with respect to $u$:

$$
\nabla_u\mathcal{L}=2\Sigma u-2\lambda u.
$$

Set the derivative equal to zero:

$$
2\Sigma u-2\lambda u=0
\quad\Longrightarrow\quad
\Sigma u=\lambda u.
$$

Thus the variance-maximizing directions are eigenvectors of the covariance matrix. The first $k$ principal components are the $k$ eigenvectors with largest eigenvalues:

$$
U_k=\begin{bmatrix}u_1&u_2&\cdots&u_k\end{bmatrix}\in\mathbb{R}^{n\times k}.
$$

Project the centered data into $k$ dimensions by

$$
Z=\widetilde{X}U_k\in\mathbb{R}^{m\times k}.
$$

Reconstruct back to the original feature space by

$$
\widehat{X}=ZU_k^T+\mu.
$$

The fraction of total variance explained by component $j$ is

$$
\text{explained variance ratio}_j=\frac{\lambda_j}{\sum_{\ell=1}^{n}\lambda_\ell},
$$

and the cumulative variance explained by the first $k$ components is

$$
\text{cumulative explained variance}(k)=\frac{\sum_{j=1}^{k}\lambda_j}{\sum_{\ell=1}^{n}\lambda_\ell}.
$$

### ICA as source separation

ICA starts from a different model. It assumes the observed vector $x\in\mathbb{R}^n$ is generated by hidden independent sources $s\in\mathbb{R}^n$ through an invertible mixing matrix $A$:

$$
x=As.
$$

The goal is to learn an unmixing matrix

$$
W=A^{-1}
$$

so that

$$
s=Wx.
$$

If $w_i^T$ is row $i$ of $W$, then recovered source $i$ is

$$
\widehat{s}_i=w_i^Tx.
$$

Because $s=Wx$, the change-of-variables formula gives

$$
p(x)=p_s(Wx)|\det W|.
$$

If the sources are independent, their joint density factorizes:

$$
p_s(Wx)=\prod_{i=1}^{n}p_s(w_i^Tx).
$$

Therefore

$$
p(x)=\prod_{i=1}^{n}p_s(w_i^Tx)\cdot |W|.
$$

For training examples $x^{(1)},\ldots,x^{(m)}$, and using the sigmoid $g$ as in the Bell-Sejnowski ICA formulation, the log-likelihood is

$$
l(W)=\sum_{i=1}^{m}\left(\sum_{j=1}^{n}\log\left(g'(w_j^Tx^{(i)})\right)+\log|W|\right).
$$

The stochastic gradient ascent update is

$$
W \leftarrow W+\alpha\left(
\begin{bmatrix}
1-2g(w_1^Tx^{(i)})\\
1-2g(w_2^Tx^{(i)})\\
\vdots\\
1-2g(w_n^Tx^{(i)})
\end{bmatrix}(x^{(i)})^T+(W^T)^{-1}
\right).
$$

PCA is about uncorrelated directions ordered by variance. ICA is about independent directions that may not be ordered by variance.

## 3. Worked Examples

### Setup

Run this once before the coded examples. The imports appear only here so the notebook can run top-to-bottom without repeated setup.

In [ ]:
import numpy as np  # Import NumPy for arrays, linear algebra, and reproducible simulations.
import matplotlib.pyplot as plt  # Import Matplotlib for all static visualizations.
from sklearn.decomposition import PCA, FastICA  # Import sklearn PCA and ICA for comparisons to scratch code.
from sklearn.datasets import load_iris, load_digits, load_wine, load_breast_cancer  # Import small built-in datasets.
from sklearn.preprocessing import StandardScaler  # Import standardization for scale-sensitive PCA examples.
from sklearn.linear_model import LogisticRegression  # Import a simple classifier for the variance-trap example.
from sklearn.model_selection import train_test_split  # Import a train-test split for fair predictive comparisons.
from sklearn.metrics import accuracy_score, mean_squared_error  # Import metrics for classification and reconstruction.
from scipy import signal  # Import signal utilities for a sawtooth waveform in the ICA example.
try:  # Try to import widgets so the final interactive experiment works in notebooks.
    from ipywidgets import interact, IntSlider  # Import interact controls when ipywidgets is installed.
except Exception:  # Fall back gracefully in plain Python environments without widgets.
    interact = None  # Store None so later code can explain that widgets are unavailable.
    IntSlider = None  # Store None so later code can avoid constructing a missing widget.
np.random.seed(229)  # Seed NumPy's legacy RNG for reproducibility in simple calls.
rng = np.random.default_rng(229)  # Create a modern random generator for reproducible synthetic data.
plt.rcParams["figure.figsize"] = (7, 4)  # Set a readable default figure size for notebook plots.
plt.rcParams["axes.grid"] = True  # Add light grids so geometric projections are easier to read.

### Data — swappable sources

These helpers create the toy, synthetic, and built-in datasets used below. Change `DATA_SOURCE` to inspect a different PCA-friendly or PCA-hostile source.

In [ ]:
DATA_SOURCE = "gaussian_2d"  # Choose one option: "gaussian_2d", "variance_trap", "iris", "digits", "wine", or "breast_cancer".

def center_columns(X):  # Define a reusable centering helper for PCA from scratch.
    mean = X.mean(axis=0)  # Compute one mean per feature column.
    X_centered = X - mean  # Subtract the feature means from every row.
    return X_centered, mean  # Return both centered data and the means needed for reconstruction.

def covariance_mle(X_centered):  # Define CS 229-style covariance with factor 1/m.
    m = X_centered.shape[0]  # Count examples so the normalization matches the lesson formula.
    return (X_centered.T @ X_centered) / m  # Compute Sigma = X^T X / m.

def pca_from_scratch(X, k):  # Define a complete PCA routine from centering through reconstruction.
    X_centered, mean = center_columns(X)  # Center the data and save the mean vector.
    Sigma = covariance_mle(X_centered)  # Compute the symmetric covariance matrix.
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)  # Use eigh because covariance matrices are symmetric.
    order = np.argsort(eigenvalues)[::-1]  # Sort eigenvalues from largest to smallest.
    eigenvalues = eigenvalues[order]  # Reorder eigenvalues by explained variance.
    eigenvectors = eigenvectors[:, order]  # Reorder eigenvectors to match the sorted eigenvalues.
    U_k = eigenvectors[:, :k]  # Keep the first k principal directions.
    Z = X_centered @ U_k  # Project centered data into k principal-coordinate dimensions.
    X_reconstructed = Z @ U_k.T + mean  # Map the k-dimensional coordinates back to feature space.
    ratios = eigenvalues / eigenvalues.sum()  # Compute explained variance ratios for all components.
    return Z, X_reconstructed, eigenvalues, eigenvectors, ratios, mean  # Return all PCA artifacts for inspection.

def make_gaussian_2d(n=240):  # Define a tilted two-dimensional Gaussian cloud.
    mean = np.array([2.0, -1.0])  # Choose a nonzero mean so centering is visibly necessary.
    covariance = np.array([[3.0, 1.8], [1.8, 1.4]])  # Choose correlated features so PCA has a tilted first axis.
    X = rng.multivariate_normal(mean, covariance, size=n)  # Sample a reproducible cloud from the chosen covariance.
    y = np.zeros(n, dtype=int)  # Store dummy labels because this dataset is unsupervised.
    return X, y  # Return features and labels in a common shape.

def make_variance_trap(n=500):  # Define data where the largest variance is not the class signal.
    y = rng.integers(0, 2, size=n)  # Create balanced binary class labels.
    high_variance_noise = rng.normal(0, 4.0, size=n)  # Create a wide feature unrelated to the label.
    low_variance_signal = (2 * y - 1) * 0.55 + rng.normal(0, 0.18, size=n)  # Create a narrow feature that predicts the label.
    X = np.column_stack([high_variance_noise, low_variance_signal])  # Combine misleading variance and useful signal.
    return X, y  # Return the trap dataset.

def load_named_dataset(name):  # Define one loader that supports all swappable data sources.
    if name == "gaussian_2d":  # Check whether the synthetic Gaussian source was requested.
        return make_gaussian_2d()  # Return the tilted cloud.
    if name == "variance_trap":  # Check whether the PCA failure source was requested.
        return make_variance_trap()  # Return the variance-trap data.
    if name == "iris":  # Check whether Iris was requested.
        data = load_iris()  # Load the Iris dataset from sklearn.
        return data.data, data.target  # Return numeric features and species labels.
    if name == "digits":  # Check whether handwritten digits were requested.
        data = load_digits()  # Load 8-by-8 digit images from sklearn.
        return data.data, data.target  # Return flattened pixels and digit labels.
    if name == "wine":  # Check whether Wine was requested.
        data = load_wine()  # Load Wine chemistry measurements.
        return data.data, data.target  # Return feature matrix and cultivar labels.
    if name == "breast_cancer":  # Check whether Breast Cancer was requested.
        data = load_breast_cancer()  # Load tumor measurement features.
        return data.data, data.target  # Return features and benign/malignant labels.
    raise ValueError(f"Unknown DATA_SOURCE: {name}")  # Fail loudly if the source name is misspelled.

X_data, y_data = load_named_dataset(DATA_SOURCE)  # Load the selected source once for exploration.
print(f"Loaded {DATA_SOURCE}: X shape = {X_data.shape}, y shape = {y_data.shape}")  # Print the dataset dimensions.

### 📖 Concept walkthrough — build each idea from scratch

Before the warm-up examples, we build PCA and ICA from scratch with tiny data you can inspect by eye. PCA will be assembled as centering, covariance, eigenvectors, projection, and explained variance; ICA will be treated as source separation after PCA-style decorrelation. Everything here uses only NumPy + Matplotlib and `_w`-suffixed variables so the walkthrough never collides with the examples below.

In [ ]:
import numpy as np  # NumPy gives us arrays, means, covariance products, and eigenvectors for the scratch build.
import matplotlib.pyplot as plt  # Matplotlib lets us check every geometric step visually instead of trusting formulas blindly.
np.random.seed(0)  # A fixed seed makes every printed number and every small jittered plot reproducible.

#### 1. PCA starts by centering the cloud and measuring covariance

PCA measures how a data cloud varies **around its own mean**, not around the origin. If we forget to center first, a nonzero mean can look like a fake direction of variation. We build a tiny correlated 2-D cloud, subtract its feature mean, then compute the sample covariance

$$
\Sigma=\frac{1}{m-1}\widetilde{X}^{\top}\widetilde{X}.
$$

In [ ]:
X_raw_w = np.array([[1.0, 1.0], [2.0, 1.4], [2.8, 2.1], [3.6, 2.5], [4.2, 3.2], [5.0, 3.6], [5.8, 4.3]])  # a tiny tilted cloud with two correlated features.
mean_w = X_raw_w.mean(axis=0)  # compute one mean per feature because PCA centers each column separately.
X_centered_w = X_raw_w - mean_w  # subtract the mean so variance is measured around the cloud's center.
print("feature means:", np.round(mean_w, 3))  # inspect the point PCA will treat as the origin.
print("first centered rows:\n", np.round(X_centered_w[:3], 3))  # verify that centering shifts rows without changing their relative spacing.
print("centered column means:", np.round(X_centered_w.mean(axis=0), 10))  # confirm the centered cloud now has mean zero.

Centering is required because variance is an average squared deviation from a mean. PCA wants directions that explain spread, so the baseline must be the data cloud's own center rather than the coordinate system's arbitrary origin.
▶ What you'll see: the centered columns have mean 0, so the cloud is now ready for variance calculations.

In [ ]:
m_w = X_centered_w.shape[0]  # count examples so the covariance normalization is explicit.
cov_w = (X_centered_w.T @ X_centered_w) / (m_w - 1)  # compute Sigma = X^T X / (m-1), the unbiased sample covariance.
print("covariance matrix:\n", np.round(cov_w, 3))  # inspect variances on the diagonal and feature co-movement off the diagonal.
print("cov(x1, x2):", round(cov_w[0, 1], 3))  # a positive off-diagonal entry means the two features rise together.

The product $\widetilde{X}^{\top}\widetilde{X}$ adds all pairwise feature products after centering. Dividing by $m-1$ turns those sums into sample variances and covariances; the diagonal entries are feature variances, and the off-diagonal entries show whether features move together.
▶ What you'll see: a positive off-diagonal covariance, matching the upward tilt in the tiny cloud.

In [ ]:
plt.figure(figsize=(5, 4))  # create one compact figure for the centered data geometry.
plt.scatter(X_centered_w[:, 0], X_centered_w[:, 1], s=80, color="slateblue")  # plot the centered coordinates PCA actually uses.
plt.axhline(0, color="gray", lw=1, ls="--")  # draw the centered horizontal axis for orientation.
plt.axvline(0, color="gray", lw=1, ls="--")  # draw the centered vertical axis for orientation.
plt.xlabel("centered feature 1")  # label the first centered feature.
plt.ylabel("centered feature 2")  # label the second centered feature.
plt.title("1: centered correlated cloud")  # title the plot with this walkthrough step.
plt.axis("equal")  # keep one unit on x equal to one unit on y so directions are honest.
plt.show()  # render the centered scatter before eigenvectors are added.

▶ What you'll see: the same data cloud now centered at $(0,0)$, with its widest spread running diagonally.

*Why it's done this way: centering removes location so PCA studies shape, and $\widetilde{X}^{\top}\widetilde{X}/(m-1)$ compresses that shape into a 2×2 covariance matrix whose entries we can inspect directly.*

#### 2. Eigenvectors of covariance are principal directions

Now PCA asks which unit direction captures the most variance. For any unit vector $u$, the projected variance is $u^{\top}\Sigma u$; maximizing that quantity with the constraint $u^{\top}u=1$ leads to $\Sigma u=\lambda u$. That is why the principal directions are eigenvectors of the covariance matrix, and the eigenvalue $\lambda$ is the variance along that eigenvector.

In [ ]:
eigvals_w, eigvecs_w = np.linalg.eigh(cov_w)  # use eigh because covariance matrices are symmetric.
order_w = np.argsort(eigvals_w)[::-1]  # sort component indices from largest variance to smallest.
eigvals_w = eigvals_w[order_w]  # reorder eigenvalues so component 1 explains the most variance.
eigvecs_w = eigvecs_w[:, order_w]  # reorder eigenvectors to match their sorted eigenvalues.
print("eigenvalues:", np.round(eigvals_w, 3))  # each eigenvalue is variance along its eigenvector.
print("eigenvectors (columns):\n", np.round(eigvecs_w, 3))  # each column is a principal direction in feature space.

▶ What you'll see: one large eigenvalue and one small eigenvalue, showing one dominant direction of spread.

In [ ]:
u1_w = eigvecs_w[:, 0]  # take the top principal direction.
u2_w = eigvecs_w[:, 1]  # take the second orthogonal principal direction.
var_u1_w = u1_w.T @ cov_w @ u1_w  # compute projected variance along the top eigenvector.
var_u2_w = u2_w.T @ cov_w @ u2_w  # compute projected variance along the second eigenvector.
diag_w = np.array([1.0, 1.0]) / np.sqrt(2)  # choose a diagonal unit direction for comparison.
var_diag_w = diag_w.T @ cov_w @ diag_w  # compute projected variance along the arbitrary diagonal direction.
print("variance along u1:", round(var_u1_w, 3))  # should equal the largest eigenvalue.
print("variance along u2:", round(var_u2_w, 3))  # should equal the smaller eigenvalue.
print("variance along [1,1]/sqrt(2):", round(var_diag_w, 3))  # compare against a hand-picked direction.

The equality $u_j^{\top}\Sigma u_j=\lambda_j$ is the numeric meaning of an eigenvalue in PCA: it is not just a linear-algebra label, it is the variance of the data after projection onto that axis. The largest $\lambda$ wins because PCA is literally maximizing projected variance.
▶ What you'll see: the top eigenvector's variance equals the largest eigenvalue and beats the weaker orthogonal direction.

In [ ]:
plt.figure(figsize=(5, 4))  # create a figure for covariance eigenvectors.
plt.scatter(X_centered_w[:, 0], X_centered_w[:, 1], s=80, alpha=0.75, color="slateblue")  # draw the centered cloud under the arrows.
for j_w in range(2):  # draw both principal directions.
    length_w = np.sqrt(eigvals_w[j_w])  # scale arrows by standard deviation so long arrows mean high variance.
    vec_w = eigvecs_w[:, j_w] * length_w  # convert unit eigenvector into a visible variance-scaled arrow.
    plt.arrow(0, 0, vec_w[0], vec_w[1], width=0.035, color=["crimson", "seagreen"][j_w], length_includes_head=True)  # plot one principal axis arrow from the mean.
    plt.text(vec_w[0] * 1.08, vec_w[1] * 1.08, f"u{j_w + 1}", color=["crimson", "seagreen"][j_w])  # label the arrow tip.
plt.axhline(0, color="gray", lw=1, ls="--")  # show the centered horizontal reference axis.
plt.axvline(0, color="gray", lw=1, ls="--")  # show the centered vertical reference axis.
plt.xlabel("centered feature 1")  # label the first centered coordinate.
plt.ylabel("centered feature 2")  # label the second centered coordinate.
plt.title("2: covariance eigenvectors as PCA directions")  # title the plot with this walkthrough step.
plt.axis("equal")  # preserve geometric angles and arrow lengths.
plt.show()  # render the principal directions over the centered cloud.

▶ What you'll see: the red top eigenvector points along the cloud's widest diagonal direction, while the green one crosses the narrow direction.

*Why it's done this way: the covariance matrix stores every direction's variance through $u^{\top}\Sigma u$, and its eigenvectors are exactly the special orthogonal axes where that variance is maximized and ordered.*

#### 3. Projection turns many features into principal coordinates

Once PCA has directions, dimensionality reduction is just a dot product. Projecting onto the top eigenvector gives one coordinate per example, $z^{(i)}=\widetilde{x}^{(i)\top}u_1$, and reconstructing from one component maps that scalar back along the same line. We also compute the explained-variance ratio $\frac{\lambda_1}{\lambda_1+\lambda_2}$ so the compression has a measurable cost.

In [ ]:
u_top_w = eigvecs_w[:, 0]  # choose the one-dimensional PCA subspace.
z1_w = X_centered_w @ u_top_w  # project each centered point onto the top eigenvector by dot product.
print("1-D PCA coordinates:", np.round(z1_w, 3))  # inspect the compressed scalar coordinate for every original point.
print("compressed shape:", z1_w.reshape(-1, 1).shape)  # show that seven 2-D points became seven 1-D coordinates.

▶ What you'll see: each 2-D point has become one signed coordinate along the main diagonal axis.

In [ ]:
X_recon_centered_w = np.outer(z1_w, u_top_w)  # reconstruct centered points by walking z units along u1.
X_recon_w = X_recon_centered_w + mean_w  # add the original mean back to return to the original coordinate system.
print("first three 1-D reconstructions:\n", np.round(X_recon_w[:3], 3))  # inspect approximate points after one-component reconstruction.
print("first three originals:\n", X_raw_w[:3])  # compare the reconstructed points to the original data.

Reconstruction cannot recover information perpendicular to $u_1$ because that coordinate was intentionally discarded. The point is that the discarded direction had low variance, so the one-dimensional approximation keeps most visible structure.
▶ What you'll see: reconstructed points lie near the originals but have been snapped onto the best-fit principal line.

In [ ]:
ratio1_w = eigvals_w[0] / eigvals_w.sum()  # compute lambda_1 divided by total variance.
ratio2_w = eigvals_w[1] / eigvals_w.sum()  # compute lambda_2 divided by total variance.
print("explained variance ratios:", np.round([ratio1_w, ratio2_w], 3))  # report variance share per principal component.
print("one-component variance kept: {:.1%}".format(ratio1_w))  # make the compression tradeoff easy to read.

The ratio $\frac{\lambda_1}{\lambda_1+\lambda_2}$ answers, "How much of the cloud's total squared spread survives if we keep only component 1?" Large ratios mean a low-dimensional view is faithful; small ratios warn that important variation is being thrown away.
▶ What you'll see: the first component keeps most of the variance because the cloud is strongly elongated.

In [ ]:
plt.figure(figsize=(5, 4))  # create a plot of projection geometry.
plt.scatter(X_centered_w[:, 0], X_centered_w[:, 1], s=80, color="lightgray", edgecolor="black", label="centered points")  # draw original centered points.
plt.scatter(X_recon_centered_w[:, 0], X_recon_centered_w[:, 1], s=80, color="crimson", label="1-D projection")  # draw their projected positions on the PCA line.
for a_w, b_w in zip(X_centered_w, X_recon_centered_w):  # connect each point to its projection.
    plt.plot([a_w[0], b_w[0]], [a_w[1], b_w[1]], color="gray", ls="--", lw=1)  # show the perpendicular information discarded by compression.
line_w = np.array([-3.5, 3.5])[:, None] * u_top_w[None, :]  # make endpoints for the top principal axis line.
plt.plot(line_w[:, 0], line_w[:, 1], color="crimson", lw=2, label="u1 axis")  # draw the one-dimensional subspace.
plt.xlabel("centered feature 1")  # label the centered x-axis.
plt.ylabel("centered feature 2")  # label the centered y-axis.
plt.legend()  # identify original points, projections, and the PCA axis.
plt.title("3: projection onto the first principal component")  # title the projection plot.
plt.axis("equal")  # keep projection distances geometrically faithful.
plt.show()  # render the one-dimensional projection.

▶ What you'll see: gray points drop onto a red PCA line, leaving only small dashed residuals perpendicular to the main direction.

*Why it's done this way: projection is a dot product because principal components are unit directions, and explained variance uses eigenvalues because each $\lambda_j$ is exactly the variance captured by component $j$.*

#### 4. ICA asks for independent non-Gaussian sources, not just uncorrelated axes

ICA starts from a different story: the observed measurements are mixtures $x=As$ of hidden sources $s$. PCA can decorrelate and rotate by variance, but uncorrelated is weaker than independent; many rotated mixtures can have zero covariance while still blending the real causes. ICA therefore looks for directions whose projected signals are as independent and non-Gaussian as possible, because Gaussian mixtures are rotation-ambiguous while non-Gaussian sources reveal preferred axes.

In [ ]:
t_w = np.linspace(0, 1, 120)  # create a short time axis for two hidden signals.
s1_w = np.sign(np.sin(2 * np.pi * 3 * t_w))  # source 1 is a non-Gaussian square-like wave.
s2_w = 2 * ((5 * t_w) % 1) - 1  # source 2 is a non-Gaussian sawtooth-like wave.
S_w = np.column_stack([s1_w, s2_w])  # store sources as rows over time and columns as hidden causes.
S_w = (S_w - S_w.mean(axis=0)) / S_w.std(axis=0)  # standardize sources so scale does not dominate the mixture.
A_w = np.array([[1.0, 0.7], [0.45, 1.0]])  # choose a fixed mixing matrix that blends both sources into both sensors.
X_mix_w = S_w @ A_w.T  # form observed mixed signals x = A s, written row-wise as S A^T.
print("mixing matrix A:\n", A_w)  # inspect how each sensor combines the hidden sources.
print("first mixed rows:\n", np.round(X_mix_w[:5], 3))  # inspect the observed signals ICA would receive.

▶ What you'll see: each observed column is a blend, not one clean original source.

In [ ]:
fig, ax_w = plt.subplots(2, 1, figsize=(7, 4), sharex=True)  # create two stacked time-series panels.
ax_w[0].plot(t_w, S_w[:, 0], label="source 1")  # draw the first hidden source.
ax_w[0].plot(t_w, S_w[:, 1], label="source 2", alpha=0.8)  # draw the second hidden source.
ax_w[0].set_title("4: hidden independent sources")  # title the source panel.
ax_w[0].legend(loc="upper right")  # label the two sources.
ax_w[1].plot(t_w, X_mix_w[:, 0], label="mixed sensor 1")  # draw the first observed mixture.
ax_w[1].plot(t_w, X_mix_w[:, 1], label="mixed sensor 2", alpha=0.8)  # draw the second observed mixture.
ax_w[1].set_title("4: observed mixtures look blended")  # title the mixture panel.
ax_w[1].set_xlabel("time")  # label the shared time axis.
ax_w[1].legend(loc="upper right")  # label the observed mixtures.
plt.tight_layout()  # reduce overlap between stacked panels.
plt.show()  # render sources versus observed mixtures.

▶ What you'll see: the mixed signals look smoother and more blended than the independent square and sawtooth sources.

In [ ]:
X_mix_centered_w = X_mix_w - X_mix_w.mean(axis=0)  # center mixed observations before whitening.
cov_mix_w = (X_mix_centered_w.T @ X_mix_centered_w) / (X_mix_centered_w.shape[0] - 1)  # compute covariance of the mixed sensors.
vals_mix_w, vecs_mix_w = np.linalg.eigh(cov_mix_w)  # eigen-decompose the mixed covariance for PCA-style whitening.
white_w = vecs_mix_w @ np.diag(1 / np.sqrt(vals_mix_w)) @ vecs_mix_w.T  # build a whitening matrix that makes covariance close to identity.
X_white_w = X_mix_centered_w @ white_w  # decorrelate and rescale the mixtures.
print("mixed covariance:\n", np.round(cov_mix_w, 3))  # show that raw sensors are correlated.
print("whitened covariance:\n", np.round(np.cov(X_white_w, rowvar=False), 3))  # show PCA whitening removes covariance only.

Whitening is often ICA's first step because it removes second-order structure: after whitening, every direction has unit variance and the covariance matrix is nearly identity. But that also shows PCA's limit — decorrelation alone does not choose the independent sources, because rotations of whitened data stay uncorrelated.
▶ What you'll see: the whitened covariance is almost the identity matrix, even though the sources are not fully recovered yet.

In [ ]:
angles_w = np.linspace(0, np.pi, 181)  # try many possible rotations of the whitened data.
scores_w = []  # store a simple non-Gaussianity score for each rotation.
for angle_w in angles_w:  # scan rotations because two whitened ICA sources differ mainly by rotation.
    R_w = np.array([[np.cos(angle_w), -np.sin(angle_w)], [np.sin(angle_w), np.cos(angle_w)]])  # make a 2-D rotation matrix.
    Y_w = X_white_w @ R_w  # rotate the whitened observations into candidate components.
    kurt_w = np.mean(Y_w ** 4, axis=0) - 3  # compute excess-kurtosis-like non-Gaussianity for each candidate component.
    scores_w.append(np.sum(np.abs(kurt_w)))  # ICA prefers rotations with strongly non-Gaussian marginal components.
best_idx_w = int(np.argmax(scores_w))  # select the rotation with the largest non-Gaussianity score.
best_angle_w = angles_w[best_idx_w]  # store the winning angle.
R_best_w = np.array([[np.cos(best_angle_w), -np.sin(best_angle_w)], [np.sin(best_angle_w), np.cos(best_angle_w)]])  # rebuild the best rotation matrix.
Y_ica_w = X_white_w @ R_best_w  # compute the illustrative ICA-style recovered components.
print("best rotation angle (radians):", round(best_angle_w, 3))  # inspect the selected unmixing rotation.
print("best non-Gaussianity score:", round(scores_w[best_idx_w], 3))  # inspect how strongly non-Gaussian the chosen components are.
print("abs corr with true sources:\n", np.round(np.abs(np.corrcoef(Y_ica_w.T, S_w.T)[:2, 2:]), 3))  # compare recovered components to hidden sources only for teaching.

This is not a full FastICA optimizer; it is a transparent two-dimensional illustration of ICA's objective. After whitening has made covariance uninformative, we search for the rotation whose components are least Gaussian, because independent real-world sources often have sharp, flat, or otherwise non-Gaussian distributions.
▶ What you'll see: the chosen rotation has components that correlate strongly with the original hidden sources, up to sign and order.

In [ ]:
fig, ax_w = plt.subplots(1, 2, figsize=(8, 3.5))  # compare PCA whitening and ICA-style rotation side by side.
ax_w[0].scatter(X_white_w[:, 0], X_white_w[:, 1], s=18, alpha=0.7, color="steelblue")  # show decorrelated whitened mixtures.
ax_w[0].set_title("4: PCA whitening decorrelates")  # label the PCA-style intermediate result.
ax_w[0].set_xlabel("white axis 1")  # label the first whitened axis.
ax_w[0].set_ylabel("white axis 2")  # label the second whitened axis.
ax_w[0].axis("equal")  # keep the whitened scatter geometry honest.
ax_w[1].scatter(Y_ica_w[:, 0], Y_ica_w[:, 1], s=18, alpha=0.7, color="darkorange")  # show the rotated ICA-style components.
ax_w[1].set_title("4: ICA rotation seeks sources")  # label the source-seeking rotation.
ax_w[1].set_xlabel("component 1")  # label the first recovered component.
ax_w[1].set_ylabel("component 2")  # label the second recovered component.
ax_w[1].axis("equal")  # keep the rotated scatter geometry honest.
plt.tight_layout()  # make the two panels fit cleanly.
plt.show()  # render the ICA comparison figure.

▶ What you'll see: whitening makes the sensor cloud round-ish, while the ICA-style rotation aligns it with sharper non-Gaussian source structure.

*Why it's done this way: PCA stops after orthogonal uncorrelated axes ordered by variance, while ICA uses non-Gaussianity to pick an unmixing rotation that better matches statistically independent causes.*

### 🟢 Basics (warm-up)

#### B1. Center a tiny matrix column by column

Goal: compute feature means and subtract them from each column.

In [ ]:
X_tiny = np.array([[1.0, 10.0], [2.0, 12.0], [3.0, 14.0]])  # Store three examples with two features.
column_means = X_tiny.mean(axis=0)  # Compute the mean of each feature column.
X_centered_tiny = X_tiny - column_means  # Subtract the column means from every row.
print("Original matrix:\n", X_tiny)  # Display the uncentered matrix.
print("Column means:", column_means)  # Display the two means being subtracted.
print("Centered matrix:\n", X_centered_tiny)  # Display the centered matrix.
print("Centered column means:", X_centered_tiny.mean(axis=0))  # Verify that centered columns have mean zero.
fig, axes = plt.subplots(1, 2, figsize=(7, 3))  # Create side-by-side panels for before and after centering.
axes[0].imshow(X_tiny, aspect="auto", cmap="Greys")  # Plot the original values as a tiny heatmap.
axes[0].set_title("Before centering")  # Label the original matrix panel.
axes[1].imshow(X_centered_tiny, aspect="auto", cmap="coolwarm")  # Plot centered values with sign-sensitive colors.
axes[1].set_title("After centering")  # Label the centered matrix panel.
for ax in axes:  # Loop over panels to add consistent axis labels.
    ax.set_xlabel("feature")  # Label columns as features.
    ax.set_ylabel("example")  # Label rows as examples.
plt.tight_layout()  # Prevent labels from overlapping.
plt.show()  # Render the before/after heatmaps.

▶ What you'll see: each feature column is shifted so its new mean is exactly zero.

👀 **Takeaway:** PCA should describe variation around the data center, not variation caused by the arbitrary origin of the coordinate system.

---

#### B2. Compute covariance between two centered features

Goal: form $\Sigma=\frac{1}{m}\widetilde{X}^T\widetilde{X}$ from centered columns.

In [ ]:
x1 = np.array([-1.0, 0.0, 1.0, 2.0])  # Store a centered-ish first feature with visible spread.
x2 = np.array([-2.0, 0.0, 2.0, 4.0])  # Store a second feature that moves with the first.
X_pair = np.column_stack([x1, x2])  # Combine the two feature columns into one data matrix.
X_pair_centered, pair_mean = center_columns(X_pair)  # Center both columns exactly.
Sigma_pair = covariance_mle(X_pair_centered)  # Compute the 2-by-2 covariance matrix with factor 1/m.
manual_cov = np.sum(X_pair_centered[:, 0] * X_pair_centered[:, 1]) / X_pair_centered.shape[0]  # Compute covariance by the scalar formula.
print("Centered features:\n", X_pair_centered)  # Show the centered feature values.
print("Manual covariance between feature 1 and feature 2:", manual_cov)  # Print the off-diagonal covariance.
print("Covariance matrix:\n", Sigma_pair)  # Print the full covariance matrix.
fig, ax = plt.subplots(figsize=(3.8, 3.4))  # Create a small covariance heatmap figure.
im = ax.imshow(Sigma_pair, cmap="Blues")  # Plot covariance magnitudes as a heatmap.
ax.set_xticks([0, 1])  # Place tick marks for the two feature columns.
ax.set_yticks([0, 1])  # Place tick marks for the two feature rows.
ax.set_xticklabels(["feature 1", "feature 2"])  # Label covariance columns.
ax.set_yticklabels(["feature 1", "feature 2"])  # Label covariance rows.
for i in range(2):  # Loop over covariance rows.
    for j in range(2):  # Loop over covariance columns.
        ax.text(j, i, f"{Sigma_pair[i, j]:.2f}", ha="center", va="center")  # Write each covariance value in its cell.
fig.colorbar(im, ax=ax, fraction=0.046)  # Add a colorbar to show magnitude.
ax.set_title("B2 covariance matrix")  # Title the heatmap.
plt.tight_layout()  # Improve spacing for labels.
plt.show()  # Render the covariance heatmap.

▶ What you'll see: the off-diagonal covariance is positive because the two centered features rise together.

👀 **Takeaway:** covariance is the average product of centered coordinates; PCA diagonalizes this matrix.

---

#### B3. Project one point onto a given unit axis

Goal: compute the one-dimensional coordinate $z=x^Tu$ and the reconstructed projection $zu$.

In [ ]:
point = np.array([3.0, 2.0])  # Store one two-dimensional point to project.
axis = np.array([1.0, 1.0]) / np.sqrt(2.0)  # Store a unit vector along the 45-degree line.
coordinate = point @ axis  # Compute the scalar projection coordinate z = x^T u.
projected_point = coordinate * axis  # Convert the scalar coordinate back to a point on the axis.
print("point x =", point)  # Print the original point.
print("unit axis u =", axis)  # Print the unit-length projection axis.
print("projected coordinate z = x^T u =", coordinate)  # Print the one-dimensional representation.
print("projection z u =", projected_point)  # Print the closest point on the chosen axis.
fig, ax = plt.subplots(figsize=(5, 5))  # Create a square plot for geometric projection.
t = np.linspace(-1, 5, 100)  # Create parameter values for drawing the axis line.
axis_line = np.outer(t, axis)  # Convert scalar positions into points along the axis.
ax.plot(axis_line[:, 0], axis_line[:, 1], color="black", label="unit axis span(u)")  # Draw the projection axis.
ax.scatter(point[0], point[1], color="crimson", s=80, label="original point x")  # Draw the original point.
ax.scatter(projected_point[0], projected_point[1], color="royalblue", s=80, label="projection z u")  # Draw the projected point.
ax.plot([point[0], projected_point[0]], [point[1], projected_point[1]], "--", color="gray", label="orthogonal drop")  # Connect point to projection.
ax.set_aspect("equal", adjustable="box")  # Use equal axes so orthogonality looks correct.
ax.set_title("B3 projection onto one unit axis")  # Title the projection plot.
ax.legend()  # Show labels for point, axis, and projection.
plt.show()  # Render the geometric projection.

▶ What you'll see: the blue projection lies on the black axis, and the dashed residual is perpendicular to that axis.

👀 **Takeaway:** PCA projection is repeated dot products against learned unit axes.

---


#### B4. Compute variance of one feature

Goal: measure the average squared deviation of one centered feature.

In [ ]:
feature_b4 = np.array([1.0, 2.0, 4.0, 5.0])  # Store one tiny feature column.
mean_b4 = feature_b4.mean()  # Compute the feature mean.
centered_b4 = feature_b4 - mean_b4  # Center the feature so deviations are relative to its mean.
variance_b4 = np.mean(centered_b4 ** 2)  # Compute PCA-style variance with factor 1/m.
print("feature:", feature_b4)  # Display original values.
print("centered feature:", centered_b4)  # Display deviations from the mean.
print("variance:", variance_b4)  # Display average squared deviation.
plt.figure(figsize=(5.5, 3.4))  # Create a small deviation plot.
plt.bar(np.arange(len(feature_b4)), centered_b4 ** 2, color="slateblue")  # Plot each squared deviation.
plt.title(f"B4 variance = {variance_b4:.2f}")  # Label the computed variance.
plt.xlabel("example")  # Label examples.
plt.ylabel("squared deviation")  # Label contribution size.
plt.show()  # Render the variance contributions.

▶ What you'll see: variance is the average height of the squared-deviation bars.

👀 **Takeaway:** PCA prefers directions with large variance, so variance is the core scalar being optimized.

---

#### B5. Unit-normalize a vector

Goal: turn a direction vector into length one before using it as an axis.

In [ ]:
v_b5 = np.array([3.0, 4.0])  # Store a non-unit direction vector.
norm_b5 = np.linalg.norm(v_b5)  # Compute its Euclidean length.
u_b5 = v_b5 / norm_b5  # Divide by length to make a unit vector.
print("original vector:", v_b5)  # Display the starting direction.
print("length:", norm_b5)  # Display the original length.
print("unit vector:", u_b5)  # Display the normalized direction.
print("unit-vector length:", np.linalg.norm(u_b5))  # Verify length one.
plt.figure(figsize=(4.5, 4.5))  # Create a square vector plot.
plt.arrow(0, 0, v_b5[0], v_b5[1], color="gray", width=0.03, length_includes_head=True, label="v")  # Draw the original vector.
plt.arrow(0, 0, u_b5[0], u_b5[1], color="royalblue", width=0.03, length_includes_head=True, label="u")  # Draw the unit vector.
plt.xlim(0, 4)  # Set horizontal bounds.
plt.ylim(0, 5)  # Set vertical bounds.
plt.gca().set_aspect("equal", adjustable="box")  # Keep geometric lengths honest.
plt.title("B5 unit-normalized direction")  # Title the plot.
plt.show()  # Render both arrows.

▶ What you'll see: the blue vector points the same way as the gray vector but has length one.

👀 **Takeaway:** PCA axes are unit vectors so projection coordinates are measured on a consistent scale.

---

#### B6. Compute explained variance ratio

Goal: convert eigenvalues into fractions of total variance explained.

In [ ]:
eigenvalues_b6 = np.array([5.0, 2.0, 1.0])  # Store three component variances.
ratios_b6 = eigenvalues_b6 / eigenvalues_b6.sum()  # Divide each eigenvalue by total variance.
cumulative_b6 = np.cumsum(ratios_b6)  # Accumulate explained variance component by component.
print("eigenvalues:", eigenvalues_b6)  # Display component variances.
print("explained variance ratios:", np.round(ratios_b6, 3))  # Display variance fractions.
print("cumulative:", np.round(cumulative_b6, 3))  # Display running total.
plt.figure(figsize=(5.5, 3.6))  # Create a small bar chart.
plt.bar([1, 2, 3], ratios_b6, color="slateblue")  # Plot per-component variance fraction.
plt.plot([1, 2, 3], cumulative_b6, marker="o", color="black")  # Overlay cumulative variance.
plt.title("B6 explained variance ratio")  # Title the plot.
plt.xlabel("component")  # Label component axis.
plt.ylabel("fraction of total variance")  # Label ratio axis.
plt.show()  # Render the variance summary.

▶ What you'll see: the first component explains the largest fraction.

👀 **Takeaway:** explained variance ratio turns eigenvalues into an interpretable compression score.

---

#### B7. Reconstruct one point from one component

Goal: map a one-dimensional PCA coordinate back into the original feature space.

In [ ]:
mean_b7 = np.array([2.0, 1.0])  # Store the original feature mean.
u_b7 = np.array([1.0, 1.0]) / np.sqrt(2.0)  # Store one unit principal axis.
z_b7 = 3.0  # Store one low-dimensional coordinate.
reconstruction_b7 = mean_b7 + z_b7 * u_b7  # Reconstruct by adding the component contribution back to the mean.
print("mean:", mean_b7)  # Display the data center.
print("coordinate z:", z_b7)  # Display the compressed coordinate.
print("reconstructed point:", np.round(reconstruction_b7, 3))  # Display the reconstructed original-space point.
plt.figure(figsize=(5, 5))  # Create a geometric reconstruction plot.
plt.scatter(mean_b7[0], mean_b7[1], color="black", s=70, label="mean")  # Draw the mean.
plt.scatter(reconstruction_b7[0], reconstruction_b7[1], color="royalblue", s=90, label="reconstruction")  # Draw the reconstructed point.
plt.plot([mean_b7[0], reconstruction_b7[0]], [mean_b7[1], reconstruction_b7[1]], color="royalblue")  # Draw the component contribution.
plt.gca().set_aspect("equal", adjustable="box")  # Preserve angles and lengths.
plt.title("B7 reconstruction from one component")  # Title the plot.
plt.legend()  # Show marker labels.
plt.show()  # Render the reconstruction.

▶ What you'll see: the reconstructed point lies on the one-dimensional component line through the mean.

👀 **Takeaway:** PCA reconstruction reverses projection only within the kept component subspace.

---

#### B8. Dot two orthonormal vectors

Goal: verify that perpendicular unit axes have dot product zero.

In [ ]:
u1_b8 = np.array([1.0, 1.0]) / np.sqrt(2.0)  # Store the first unit direction.
u2_b8 = np.array([1.0, -1.0]) / np.sqrt(2.0)  # Store a perpendicular unit direction.
dot_b8 = u1_b8 @ u2_b8  # Compute the dot product between directions.
print("u1 length:", np.linalg.norm(u1_b8))  # Verify first vector length.
print("u2 length:", np.linalg.norm(u2_b8))  # Verify second vector length.
print("u1 dot u2:", round(float(dot_b8), 6))  # Verify orthogonality.
plt.figure(figsize=(4.5, 4.5))  # Create a vector plot.
plt.arrow(0, 0, u1_b8[0], u1_b8[1], color="royalblue", width=0.02, length_includes_head=True)  # Draw first axis.
plt.arrow(0, 0, u2_b8[0], u2_b8[1], color="crimson", width=0.02, length_includes_head=True)  # Draw second axis.
plt.xlim(-1, 1)  # Set horizontal bounds.
plt.ylim(-1, 1)  # Set vertical bounds.
plt.gca().set_aspect("equal", adjustable="box")  # Preserve perpendicular geometry.
plt.title("B8 orthonormal axes")  # Title the plot.
plt.show()  # Render the axes.

▶ What you'll see: both vectors have length one and dot product zero.

👀 **Takeaway:** PCA components form orthonormal axes, so their coordinates do not overlap.

---

#### B9. Total variance as covariance trace

Goal: add diagonal covariance entries to get total feature variance.

In [ ]:
Sigma_b9 = np.array([[3.0, 0.8], [0.8, 1.5]])  # Store one covariance matrix.
diagonal_b9 = np.diag(Sigma_b9)  # Extract feature variances from the diagonal.
trace_b9 = np.trace(Sigma_b9)  # Sum the diagonal entries.
print("covariance matrix:\n", Sigma_b9)  # Display covariance matrix.
print("diagonal variances:", diagonal_b9)  # Display per-feature variances.
print("total variance = trace:", trace_b9)  # Display total variance.
plt.figure(figsize=(3.8, 3.4))  # Create a heatmap figure.
plt.imshow(Sigma_b9, cmap="Blues")  # Plot covariance entries.
plt.colorbar(fraction=0.046)  # Add a color scale.
plt.title("B9 covariance trace")  # Title the heatmap.
plt.show()  # Render the covariance matrix.

▶ What you'll see: the trace adds only the variance entries on the diagonal.

👀 **Takeaway:** total variance is preserved by rotation and equals the sum of PCA eigenvalues.

---

#### B10. Eigenvalues of a 2×2 covariance matrix by formula

Goal: compute two eigenvalues from trace and determinant.

In [ ]:
Sigma_b10 = np.array([[3.0, 1.0], [1.0, 3.0]])  # Store a symmetric 2-by-2 covariance matrix.
trace_b10 = np.trace(Sigma_b10)  # Compute a + d.
determinant_b10 = np.linalg.det(Sigma_b10)  # Compute ad - bc.
discriminant_b10 = trace_b10 ** 2 - 4.0 * determinant_b10  # Compute the quadratic discriminant.
eigenvalues_b10 = np.array([(trace_b10 + np.sqrt(discriminant_b10)) / 2.0, (trace_b10 - np.sqrt(discriminant_b10)) / 2.0])  # Apply the 2-by-2 eigenvalue formula.
print("trace:", trace_b10)  # Display the trace.
print("determinant:", round(float(determinant_b10), 3))  # Display the determinant.
print("eigenvalues:", np.round(eigenvalues_b10, 3))  # Display component variances.
plt.figure(figsize=(4.5, 3.2))  # Create a small eigenvalue plot.
plt.bar(["lambda 1", "lambda 2"], eigenvalues_b10, color="slateblue")  # Plot the two eigenvalues.
plt.title("B10 eigenvalues from trace and determinant")  # Title the plot.
plt.ylabel("variance")  # Label eigenvalue axis.
plt.show()  # Render component variances.

▶ What you'll see: the larger eigenvalue corresponds to the higher-variance principal direction.

👀 **Takeaway:** eigenvalues are the variances PCA assigns to its component axes.

---

### 🟡 Easy

#### E1. Hand-compute eigenvectors of a 2×2 covariance matrix

**Problem.** For

$$
\Sigma=\begin{bmatrix}3&1\\1&3\end{bmatrix},
$$

compute the eigenvalues and eigenvectors by hand. Interpret the principal axes.

**Step 1: write the eigenvalue equation.** An eigenpair satisfies

$$
\Sigma u=\lambda u.
$$

Equivalently,

$$
(\Sigma-\lambda I)u=0.
$$

Here

$$
\Sigma-\lambda I
=
\begin{bmatrix}3&1\\1&3\end{bmatrix}
-
\begin{bmatrix}\lambda&0\\0&\lambda\end{bmatrix}
=
\begin{bmatrix}3-\lambda&1\\1&3-\lambda\end{bmatrix}.
$$

**Step 2: compute the characteristic polynomial.** Nonzero solutions require

$$
\det(\Sigma-\lambda I)=0.
$$

Compute the determinant:

$$
\det\begin{bmatrix}3-\lambda&1\\1&3-\lambda\end{bmatrix}
=(3-\lambda)(3-\lambda)-1\cdot 1.
$$

Expand:

$$
(3-\lambda)^2-1=0.
$$

Use difference of squares:

$$
(3-\lambda)^2-1^2=\big((3-\lambda)-1\big)\big((3-\lambda)+1\big)=0.
$$

Simplify both factors:

$$
(2-\lambda)(4-\lambda)=0.
$$

Therefore

$$
\lambda_1=4,
\qquad
\lambda_2=2.
$$

**Step 3: find the eigenvector for $\lambda_1=4$.** Substitute $\lambda=4$:

$$
\Sigma-4I
=
\begin{bmatrix}3-4&1\\1&3-4\end{bmatrix}
=
\begin{bmatrix}-1&1\\1&-1\end{bmatrix}.
$$

Let $u=\begin{bmatrix}a\\b\end{bmatrix}$. Then

$$
\begin{bmatrix}-1&1\\1&-1\end{bmatrix}
\begin{bmatrix}a\\b\end{bmatrix}
=
\begin{bmatrix}-a+b\\a-b\end{bmatrix}
=
\begin{bmatrix}0\\0\end{bmatrix}.
$$

The equations are

$$
-a+b=0,
\qquad
 a-b=0.
$$

Both say

$$
b=a.
$$

Choose $a=1$, $b=1$:

$$
v_1=\begin{bmatrix}1\\1\end{bmatrix}.
$$

Normalize:

$$
\|v_1\|=\sqrt{1^2+1^2}=\sqrt{2},
\qquad
u_1=\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}.
$$

**Step 4: find the eigenvector for $\lambda_2=2$.** Substitute $\lambda=2$:

$$
\Sigma-2I
=
\begin{bmatrix}1&1\\1&1\end{bmatrix}.
$$

Again let $u=\begin{bmatrix}a\\b\end{bmatrix}$. Then

$$
\begin{bmatrix}1&1\\1&1\end{bmatrix}
\begin{bmatrix}a\\b\end{bmatrix}
=
\begin{bmatrix}a+b\\a+b\end{bmatrix}
=
\begin{bmatrix}0\\0\end{bmatrix}.
$$

So

$$
a+b=0
\quad\Longrightarrow\quad
b=-a.
$$

Choose $a=1$, $b=-1$:

$$
v_2=\begin{bmatrix}1\\-1\end{bmatrix}.
$$

Normalize:

$$
\|v_2\|=\sqrt{1^2+(-1)^2}=\sqrt{2},
\qquad
u_2=\frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}.
$$

**Step 5: interpret as PCA axes.** The larger eigenvalue is $4$, so the first principal component is the line $x_2=x_1$. The second component is the orthogonal line $x_2=-x_1$.

The variance explained ratios are

$$
\frac{\lambda_1}{\lambda_1+\lambda_2}=\frac{4}{4+2}=\frac{2}{3},
\qquad
\frac{\lambda_2}{\lambda_1+\lambda_2}=\frac{2}{6}=\frac{1}{3}.
$$

**Boxed answer.**

$$
\boxed{
\lambda_1=4,
\quad
u_1=\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix},
\quad
\lambda_2=2,
\quad
u_2=\frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}
}
$$

and PC1 explains $\boxed{2/3}$ of the total variance.

---

#### E2. PCA projection by hand for four centered 2-D points

**Problem.** Consider four already-centered points

$$
(2,0),\quad (0,2),\quad (-2,0),\quad (0,-2).
$$

Project them onto the unit axis

$$
u=\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}.
$$

Compute each one-dimensional coordinate and reconstruction.

**Step 1: write the projection formula.** For a centered point $x$, the coordinate is

$$
z=x^Tu.
$$

The one-dimensional reconstruction inside the original plane is

$$
\widehat{x}=zu.
$$

**Step 2: project $x^{(1)}=(2,0)$.**

$$
z_1=\begin{bmatrix}2&0\end{bmatrix}\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=\frac{2}{\sqrt{2}}+0
=\sqrt{2}.
$$

Then

$$
\widehat{x}^{(1)}=z_1u
=\sqrt{2}\cdot\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=\begin{bmatrix}1\\1\end{bmatrix}.
$$

**Step 3: project $x^{(2)}=(0,2)$.**

$$
z_2=\begin{bmatrix}0&2\end{bmatrix}\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=0+\frac{2}{\sqrt{2}}
=\sqrt{2}.
$$

Thus

$$
\widehat{x}^{(2)}=\sqrt{2}\cdot\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=\begin{bmatrix}1\\1\end{bmatrix}.
$$

**Step 4: project $x^{(3)}=(-2,0)$.**

$$
z_3=\begin{bmatrix}-2&0\end{bmatrix}\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=-\frac{2}{\sqrt{2}}+0
=-\sqrt{2}.
$$

Thus

$$
\widehat{x}^{(3)}=-\sqrt{2}\cdot\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=\begin{bmatrix}-1\\-1\end{bmatrix}.
$$

**Step 5: project $x^{(4)}=(0,-2)$.**

$$
z_4=\begin{bmatrix}0&-2\end{bmatrix}\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=0-\frac{2}{\sqrt{2}}
=-\sqrt{2}.
$$

Thus

$$
\widehat{x}^{(4)}=-\sqrt{2}\cdot\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}
=\begin{bmatrix}-1\\-1\end{bmatrix}.
$$

**Step 6: measure reconstruction error for one point.** For $x^{(1)}=(2,0)$,

$$
x^{(1)}-\widehat{x}^{(1)}=\begin{bmatrix}2\\0\end{bmatrix}-\begin{bmatrix}1\\1\end{bmatrix}
=\begin{bmatrix}1\\-1\end{bmatrix}.
$$

The squared error is

$$
\|x^{(1)}-\widehat{x}^{(1)}\|^2=1^2+(-1)^2=2.
$$

By symmetry every point has squared reconstruction error $2$.

**Boxed answer.**

$$
\boxed{z=(\sqrt{2},\sqrt{2},-\sqrt{2},-\sqrt{2})}
$$

and

$$
\boxed{
\widehat{x}^{(1)}=\widehat{x}^{(2)}=(1,1),
\qquad
\widehat{x}^{(3)}=\widehat{x}^{(4)}=(-1,-1).
}
$$

---

#### E3. PCA on a tilted Gaussian cloud

Goal: build PCA from scratch on a correlated 2-D cloud and draw the principal axes.

In [ ]:
X_gauss, _ = make_gaussian_2d(n=260)  # Generate a tilted two-dimensional Gaussian dataset.
X_gauss_centered, gauss_mean = center_columns(X_gauss)  # Center the cloud before computing covariance.
Sigma_gauss = covariance_mle(X_gauss_centered)  # Compute the CS 229 covariance matrix.
eigvals_gauss, eigvecs_gauss = np.linalg.eigh(Sigma_gauss)  # Decompose the symmetric covariance matrix.
order_gauss = np.argsort(eigvals_gauss)[::-1]  # Sort eigenvalues from largest variance to smallest.
eigvals_gauss = eigvals_gauss[order_gauss]  # Reorder eigenvalues by importance.
eigvecs_gauss = eigvecs_gauss[:, order_gauss]  # Reorder eigenvectors to match eigenvalues.
ratios_gauss = eigvals_gauss / eigvals_gauss.sum()  # Convert eigenvalues into variance-explained ratios.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # Create panels for covariance and principal axes.
im = axes[0].imshow(Sigma_gauss, cmap="Blues")  # Visualize covariance entries as a heatmap.
axes[0].set_title("Centered covariance")  # Title the covariance panel.
for i in range(2):  # Loop over covariance rows.
    for j in range(2):  # Loop over covariance columns.
        axes[0].text(j, i, f"{Sigma_gauss[i, j]:.2f}", ha="center", va="center")  # Annotate covariance values.
fig.colorbar(im, ax=axes[0], fraction=0.046)  # Add a colorbar for covariance magnitude.
axes[1].scatter(X_gauss[:, 0], X_gauss[:, 1], s=20, alpha=0.55)  # Draw the original tilted cloud.
axes[1].scatter(gauss_mean[0], gauss_mean[1], color="black", s=70, label="mean")  # Mark the data mean.
for j, color in enumerate(["crimson", "royalblue"]):  # Loop over the first and second principal axes.
    direction = eigvecs_gauss[:, j]  # Select principal direction j.
    length = 2.5 * np.sqrt(eigvals_gauss[j])  # Scale the arrow by standard deviation along that direction.
    axes[1].arrow(gauss_mean[0], gauss_mean[1], length * direction[0], length * direction[1], color=color, width=0.03, length_includes_head=True, label=f"PC{j+1}: {ratios_gauss[j]:.1%}")  # Draw a variance-scaled PC arrow.
axes[1].set_aspect("equal", adjustable="box")  # Use equal scales so angles are meaningful.
axes[1].set_title("Principal axes over data")  # Title the scatter panel.
axes[1].legend()  # Show labels with variance explained.
plt.tight_layout()  # Improve spacing between panels.
plt.show()  # Render the PCA process and result.

▶ What you'll see: the first red principal axis follows the long direction of the tilted cloud and explains most of the variance.

👀 **Takeaway:** PCA rotates the coordinate system so the covariance matrix becomes diagonal in the principal-component basis.

---

#### E4. Explained variance on Iris

Goal: compute PCA on four Iris measurements, plot the scree curve, and view a 2-D projection.

In [ ]:
iris = load_iris()  # Load Iris data with four numeric flower measurements.
X_iris = iris.data  # Store the feature matrix.
y_iris = iris.target  # Store species labels for coloring after unsupervised PCA.
X_iris_scaled = StandardScaler().fit_transform(X_iris)  # Standardize features so centimeters with different spreads are comparable.
Z_iris, _, eigvals_iris, eigvecs_iris, ratios_iris, _ = pca_from_scratch(X_iris_scaled, k=2)  # Run scratch PCA and keep a 2-D projection.
cumulative_iris = np.cumsum(ratios_iris)  # Compute cumulative variance explained as components are added.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))  # Create side-by-side panels for variance and projection.
axes[0].bar(np.arange(1, len(ratios_iris) + 1), ratios_iris, alpha=0.75, label="individual")  # Plot individual explained variance ratios.
axes[0].plot(np.arange(1, len(ratios_iris) + 1), cumulative_iris, marker="o", color="black", label="cumulative")  # Plot cumulative explained variance.
axes[0].set_xlabel("principal component")  # Label component number on the horizontal axis.
axes[0].set_ylabel("variance explained")  # Label the variance fraction on the vertical axis.
axes[0].set_ylim(0, 1.05)  # Keep the full 0-to-1 variance range visible.
axes[0].set_title("Iris scree and cumulative variance")  # Title the scree panel.
axes[0].legend()  # Show individual and cumulative labels.
scatter = axes[1].scatter(Z_iris[:, 0], Z_iris[:, 1], c=y_iris, cmap="viridis", s=35, alpha=0.85)  # Plot the first two PCA coordinates colored by species.
axes[1].set_xlabel("PC1")  # Label the first principal coordinate.
axes[1].set_ylabel("PC2")  # Label the second principal coordinate.
axes[1].set_title("Iris projected to two PCs")  # Title the projection panel.
legend = axes[1].legend(*scatter.legend_elements(), title="species")  # Build a species legend from color classes.
axes[1].add_artist(legend)  # Add the legend to the projection panel.
plt.tight_layout()  # Improve panel spacing.
plt.show()  # Render the scree curve and projection.
print("Explained variance ratios:", np.round(ratios_iris, 3))  # Print variance ratios for exact reading.
print("Cumulative variance:", np.round(cumulative_iris, 3))  # Print cumulative ratios for exact reading.

▶ What you'll see: the first two standardized Iris PCs capture most, but not all, of the variance and reveal strong species structure.

👀 **Takeaway:** explained variance tells you how much geometric spread is retained, not whether all class information is perfectly preserved.

---

#### E5. Reconstruction from 1, 2, and 3 principal components on digits

Goal: compress small digit images and reconstruct them from a few principal components.

In [ ]:
digits = load_digits()  # Load 8-by-8 grayscale digit images.
X_digits = digits.data / 16.0  # Scale pixel intensities from 0..16 into 0..1.
y_digits = digits.target  # Store digit labels for selecting examples.
component_options = [1, 2, 3]  # Choose small reconstruction dimensions for an easy comparison.
reconstructions = []  # Prepare a list to store reconstructed digit matrices.
errors = []  # Prepare a list to store mean squared reconstruction errors.
for k in component_options:  # Loop over reconstruction dimensionalities.
    _, X_hat_k, _, _, ratios_k, _ = pca_from_scratch(X_digits, k=k)  # Fit scratch PCA and reconstruct from k components.
    reconstructions.append(X_hat_k)  # Store the reconstructed feature matrix.
    errors.append(mean_squared_error(X_digits, X_hat_k))  # Store the average pixel-level squared error.
_, _, eigvals_digits, _, ratios_digits, _ = pca_from_scratch(X_digits, k=10)  # Fit once with enough components to inspect variance ratios.
fig, axes = plt.subplots(2, 4, figsize=(9, 4.8))  # Create a grid for original and reconstructed digit images.
example_index = 7  # Choose one digit example to reconstruct repeatedly.
axes[0, 0].imshow(X_digits[example_index].reshape(8, 8), cmap="gray_r")  # Plot the original digit image.
axes[0, 0].set_title(f"original {y_digits[example_index]}")  # Label the original image with its digit class.
axes[1, 0].axis("off")  # Leave the lower-left cell empty for visual balance.
for col, k in enumerate(component_options, start=1):  # Loop over reconstruction panels.
    axes[0, col].imshow(reconstructions[col - 1][example_index].reshape(8, 8), cmap="gray_r")  # Plot reconstructed pixels for this k.
    axes[0, col].set_title(f"k={k}")  # Label the reconstruction dimensionality.
    axes[1, col].bar(["MSE"], [errors[col - 1]], color="gray")  # Plot the reconstruction error as a tiny bar chart.
    axes[1, col].set_ylim(0, max(errors) * 1.2)  # Use a common error scale for fair comparison.
    axes[1, col].set_title(f"error={errors[col - 1]:.3f}")  # Print the exact error in the title.
for ax in axes.ravel():  # Loop over all panels.
    ax.set_xticks([])  # Hide x ticks for image cleanliness.
    ax.set_yticks([])  # Hide y ticks for image cleanliness.
plt.tight_layout()  # Prevent panel overlap.
plt.show()  # Render original, reconstructions, and errors.
print("First 10 digit PCA variance ratios:", np.round(ratios_digits[:10], 3))  # Print leading variance ratios.

▶ What you'll see: one component gives a rough blur, while adding components lowers reconstruction error and restores more digit structure.

👀 **Takeaway:** PCA compression trades fewer coordinates for more reconstruction error; the tradeoff is measurable.

---

### 🔴 Advanced

#### A1. PCA failure: high variance is not always predictive

Goal: show a dataset where PC1 captures a large noisy direction and discards the low-variance class signal.

In [ ]:
X_trap, y_trap = make_variance_trap(n=700)  # Generate a dataset with high-variance noise and low-variance label signal.
Z_trap_1, _, eigvals_trap, eigvecs_trap, ratios_trap, trap_mean = pca_from_scratch(X_trap, k=1)  # Reduce the raw trap data to one PC.
X_train, X_test, y_train, y_test = train_test_split(X_trap, y_trap, test_size=0.35, random_state=229, stratify=y_trap)  # Split original features into train and test sets.
Z_train, Z_test, zy_train, zy_test = train_test_split(Z_trap_1, y_trap, test_size=0.35, random_state=229, stratify=y_trap)  # Split the one-PC features with the same random seed.
clf_full = LogisticRegression().fit(X_train, y_train)  # Fit logistic regression using both original features.
clf_pc1 = LogisticRegression().fit(Z_train, zy_train)  # Fit logistic regression using only PC1.
acc_full = accuracy_score(y_test, clf_full.predict(X_test))  # Measure accuracy with both features.
acc_pc1 = accuracy_score(zy_test, clf_pc1.predict(Z_test))  # Measure accuracy after projecting to PC1.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))  # Create panels for geometry and accuracy.
scatter = axes[0].scatter(X_trap[:, 0], X_trap[:, 1], c=y_trap, cmap="coolwarm", s=22, alpha=0.7)  # Plot trap data colored by class.
for j, color in enumerate(["black", "green"]):  # Loop over the first two principal directions.
    direction = eigvecs_trap[:, j]  # Select PC direction j.
    length = 2.5 * np.sqrt(eigvals_trap[j])  # Scale the arrow by standard deviation.
    axes[0].arrow(trap_mean[0], trap_mean[1], length * direction[0], length * direction[1], color=color, width=0.05, length_includes_head=True, label=f"PC{j+1}: {ratios_trap[j]:.1%}")  # Draw the PC axis.
axes[0].set_xlabel("wide noisy feature")  # Label the misleading high-variance axis.
axes[0].set_ylabel("narrow class signal")  # Label the predictive low-variance axis.
axes[0].set_title("Variance trap geometry")  # Title the failure geometry.
axes[0].legend()  # Show principal-axis labels.
axes[1].bar(["both features", "PC1 only"], [acc_full, acc_pc1], color=["gray", "crimson"])  # Compare predictive accuracy.
axes[1].set_ylim(0, 1.05)  # Use the full accuracy range.
axes[1].set_ylabel("test accuracy")  # Label the accuracy metric.
axes[1].set_title("Prediction can suffer after PCA")  # Title the performance comparison.
plt.tight_layout()  # Improve spacing.
plt.show()  # Render geometry and accuracy panels.
print(f"Accuracy with both features: {acc_full:.3f}")  # Print full-feature accuracy.
print(f"Accuracy with PC1 only: {acc_pc1:.3f}")  # Print PC1-only accuracy.

▶ What you'll see: PC1 follows the broad horizontal noise direction, while the class separation is mostly vertical.

👀 **Takeaway:** PCA is unsupervised; maximum variance is not guaranteed to be the most predictive direction.

---

#### A2. Scaling changes PCA directions

Goal: compare PCA on raw Wine features versus standardized Wine features.

In [ ]:
wine = load_wine()  # Load Wine features with heterogeneous physical units.
X_wine = wine.data  # Store raw Wine measurements.
feature_names_wine = wine.feature_names  # Store feature names for loading interpretation.
Z_wine_raw, _, _, eigvecs_raw, ratios_raw, _ = pca_from_scratch(X_wine, k=2)  # Run PCA directly on raw units.
X_wine_scaled = StandardScaler().fit_transform(X_wine)  # Standardize each feature to mean zero and unit variance.
Z_wine_scaled, _, _, eigvecs_scaled, ratios_scaled, _ = pca_from_scratch(X_wine_scaled, k=2)  # Run PCA on standardized features.
raw_loadings = eigvecs_raw[:, 0]  # Extract raw PC1 loadings.
scaled_loadings = eigvecs_scaled[:, 0]  # Extract standardized PC1 loadings.
top_raw = np.argsort(np.abs(raw_loadings))[::-1][:6]  # Find the largest raw-unit PC1 contributors.
top_scaled = np.argsort(np.abs(scaled_loadings))[::-1][:6]  # Find the largest standardized PC1 contributors.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # Create a four-panel comparison.
axes[0, 0].scatter(Z_wine_raw[:, 0], Z_wine_raw[:, 1], c=wine.target, cmap="viridis", s=35)  # Plot raw PCA projection.
axes[0, 0].set_title(f"Raw PCA: PC1+PC2 = {(ratios_raw[0]+ratios_raw[1]):.1%}")  # Title raw variance retained.
axes[0, 0].set_xlabel("raw PC1")  # Label raw PC1.
axes[0, 0].set_ylabel("raw PC2")  # Label raw PC2.
axes[0, 1].scatter(Z_wine_scaled[:, 0], Z_wine_scaled[:, 1], c=wine.target, cmap="viridis", s=35)  # Plot standardized PCA projection.
axes[0, 1].set_title(f"Scaled PCA: PC1+PC2 = {(ratios_scaled[0]+ratios_scaled[1]):.1%}")  # Title scaled variance retained.
axes[0, 1].set_xlabel("scaled PC1")  # Label scaled PC1.
axes[0, 1].set_ylabel("scaled PC2")  # Label scaled PC2.
axes[1, 0].barh([feature_names_wine[i] for i in top_raw][::-1], raw_loadings[top_raw][::-1], color="gray")  # Plot largest raw PC1 loadings.
axes[1, 0].set_title("Top raw PC1 loadings")  # Title raw loading panel.
axes[1, 1].barh([feature_names_wine[i] for i in top_scaled][::-1], scaled_loadings[top_scaled][::-1], color="royalblue")  # Plot largest standardized PC1 loadings.
axes[1, 1].set_title("Top scaled PC1 loadings")  # Title scaled loading panel.
plt.tight_layout()  # Improve spacing across four panels.
plt.show()  # Render projections and loading bars.
print("Raw PC1 variance ratio:", round(ratios_raw[0], 3))  # Print raw PC1 variance ratio.
print("Scaled PC1 variance ratio:", round(ratios_scaled[0], 3))  # Print scaled PC1 variance ratio.

▶ What you'll see: raw PCA is dominated by large-scale measurements, while standardized PCA distributes importance across chemically meaningful features.

👀 **Takeaway:** if feature units are arbitrary or incomparable, standardize before PCA; otherwise the largest-unit feature can dominate the covariance matrix.

---

#### A3. ICA source separation for mixed signals

Goal: mix three independent time signals and recover them with ICA.

In [ ]:
n_samples = 1200  # Choose enough time points to make source separation stable.
time = np.linspace(0, 8, n_samples)  # Create a time axis for signal generation.
s1 = np.sin(2 * time)  # Create a smooth sinusoidal source.
s2 = np.sign(np.sin(3 * time))  # Create a square-wave source with abrupt changes.
s3 = signal.sawtooth(2 * np.pi * time / 2.5)  # Create a sawtooth source with asymmetric shape.
S_true = np.column_stack([s1, s2, s3])  # Combine the three source signals as columns.
S_true = StandardScaler().fit_transform(S_true)  # Standardize sources so scale does not affect comparison.
A_mix = np.array([[1.0, 0.5, 0.2], [0.4, 1.0, 0.5], [0.2, 0.6, 1.0]])  # Define an invertible mixing matrix.
X_mixed = S_true @ A_mix.T  # Mix sources into observed signals.
X_mixed += 0.03 * rng.normal(size=X_mixed.shape)  # Add small noise so the example is realistic.
ica = FastICA(n_components=3, whiten="unit-variance", random_state=229, max_iter=1000, tol=1e-4)  # Configure ICA for three recovered components.
S_recovered = ica.fit_transform(X_mixed)  # Fit ICA and recover estimated independent components.
correlation = np.corrcoef(S_true.T, S_recovered.T)[:3, 3:]  # Compute source-to-recovery correlations.
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=False)  # Create stacked panels for sources, mixtures, recoveries, and correlations.
axes[0].plot(time, S_true)  # Plot the original hidden sources.
axes[0].set_title("True independent sources")  # Title the source panel.
axes[1].plot(time, X_mixed)  # Plot the observed mixtures.
axes[1].set_title("Observed mixtures")  # Title the mixture panel.
axes[2].plot(time, S_recovered)  # Plot ICA-recovered components.
axes[2].set_title("ICA recovered components")  # Title the recovered panel.
im = axes[3].imshow(np.abs(correlation), cmap="magma", vmin=0, vmax=1)  # Plot absolute correlations because ICA sign flips are arbitrary.
axes[3].set_title("|correlation(true source, recovered component)|")  # Title the matching heatmap.
axes[3].set_xlabel("recovered component")  # Label recovered components on columns.
axes[3].set_ylabel("true source")  # Label true sources on rows.
for i in range(3):  # Loop over true sources.
    for j in range(3):  # Loop over recovered components.
        axes[3].text(j, i, f"{abs(correlation[i, j]):.2f}", ha="center", va="center", color="white")  # Annotate correlation strength.
fig.colorbar(im, ax=axes[3], fraction=0.03)  # Add a colorbar for correlation magnitude.
plt.tight_layout()  # Improve vertical spacing.
plt.show()  # Render the ICA separation workflow.

▶ What you'll see: the mixtures look blended, but each recovered ICA component strongly correlates with one true source up to sign and order.

👀 **Takeaway:** ICA cares about statistical independence; recovered components can be permuted and sign-flipped without changing the solution.

---

#### A4. PCA denoising vs information loss

Goal: use PCA reconstruction to remove pixel noise while watching reconstruction error as components increase.

In [ ]:
digits_denoise = load_digits()  # Load the digit image dataset again for denoising.
X_clean = digits_denoise.data / 16.0  # Scale clean pixels to the 0..1 range.
noise = rng.normal(0, 0.28, size=X_clean.shape)  # Generate Gaussian pixel noise.
X_noisy = np.clip(X_clean + noise, 0.0, 1.0)  # Add noise and clip to valid image intensity range.
k_values = [2, 5, 10, 20, 40]  # Choose a range of PCA dimensions for denoising.
noisy_errors = []  # Prepare a list for reconstruction error against clean images.
reconstructed_by_k = {}  # Prepare a dictionary of reconstructed images by k.
for k in k_values:  # Loop over candidate component counts.
    _, X_denoised_k, _, _, _, _ = pca_from_scratch(X_noisy, k=k)  # Fit PCA on noisy data and reconstruct from k components.
    X_denoised_k = np.clip(X_denoised_k, 0.0, 1.0)  # Clip reconstructed pixels to image intensity range.
    reconstructed_by_k[k] = X_denoised_k  # Store the denoised reconstruction.
    noisy_errors.append(mean_squared_error(X_clean, X_denoised_k))  # Measure error relative to the clean images.
fig, axes = plt.subplots(3, 6, figsize=(11, 5.5))  # Create an image grid and an error panel.
image_indices = [0, 10]  # Choose two example digits to visualize.
for row, idx in enumerate(image_indices):  # Loop over selected digit examples.
    axes[row, 0].imshow(X_clean[idx].reshape(8, 8), cmap="gray_r")  # Show the clean digit.
    axes[row, 0].set_title("clean")  # Label the clean panel.
    axes[row, 1].imshow(X_noisy[idx].reshape(8, 8), cmap="gray_r")  # Show the noisy digit.
    axes[row, 1].set_title("noisy")  # Label the noisy panel.
    for col, k in enumerate(k_values[:4], start=2):  # Show several denoised reconstructions.
        axes[row, col].imshow(reconstructed_by_k[k][idx].reshape(8, 8), cmap="gray_r")  # Plot the denoised image for this k.
        axes[row, col].set_title(f"k={k}")  # Label the component count.
axes[2, 0].axis("off")  # Turn off unused lower-left panel.
axes[2, 1].axis("off")  # Turn off unused lower-middle panel.
axes[2, 2].plot(k_values, noisy_errors, marker="o", color="crimson")  # Plot denoising error versus number of components.
axes[2, 2].set_title("MSE to clean images")  # Title the error curve.
axes[2, 2].set_xlabel("components")  # Label the horizontal axis.
axes[2, 2].set_ylabel("MSE")  # Label the vertical axis.
for col in [3, 4, 5]:  # Loop over remaining unused lower panels.
    axes[2, col].axis("off")  # Hide unused panels for a cleaner figure.
for ax in axes.ravel():  # Loop over every subplot.
    ax.set_xticks([])  # Hide x ticks for images and compact plots.
    ax.set_yticks([])  # Hide y ticks for images and compact plots.
plt.tight_layout()  # Improve grid spacing.
plt.show()  # Render denoising examples and error curve.
print("Denoising MSE by k:", dict(zip(k_values, np.round(noisy_errors, 4))))  # Print exact denoising errors.

▶ What you'll see: very small $k$ oversmooths digits, moderate $k$ removes noise well, and large $k$ begins to keep more noisy detail.

👀 **Takeaway:** PCA denoising works because low-variance directions often contain noise, but too few components also erase real structure.

---

#### A5. Capstone: reduce a real dataset then inspect loadings

Goal: run a complete PCA workflow on Breast Cancer data: standardize, reduce, inspect loadings, and name limitations.

In [ ]:
cancer = load_breast_cancer()  # Load a real tabular medical dataset from sklearn.
X_cancer = cancer.data  # Store tumor measurement features.
y_cancer = cancer.target  # Store benign/malignant labels for visualization only.
feature_names_cancer = np.array(cancer.feature_names)  # Store feature names as an array for indexing.
X_cancer_scaled = StandardScaler().fit_transform(X_cancer)  # Standardize features before PCA because units differ.
Z_cancer, X_cancer_recon, eigvals_cancer, eigvecs_cancer, ratios_cancer, _ = pca_from_scratch(X_cancer_scaled, k=2)  # Run scratch PCA with two components.
cumulative_cancer = np.cumsum(ratios_cancer)  # Compute cumulative explained variance for model selection.
pc1_loadings = eigvecs_cancer[:, 0]  # Extract PC1 loadings across original features.
pc2_loadings = eigvecs_cancer[:, 1]  # Extract PC2 loadings across original features.
top_pc1 = np.argsort(np.abs(pc1_loadings))[::-1][:8]  # Find strongest PC1 feature loadings.
top_pc2 = np.argsort(np.abs(pc2_loadings))[::-1][:8]  # Find strongest PC2 feature loadings.
reconstruction_mse_2d = mean_squared_error(X_cancer_scaled, X_cancer_recon)  # Compute scaled-space reconstruction error from two PCs.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))  # Create a capstone dashboard.
scatter = axes[0, 0].scatter(Z_cancer[:, 0], Z_cancer[:, 1], c=y_cancer, cmap="coolwarm", s=28, alpha=0.8)  # Plot 2-D PCA coordinates colored by diagnosis.
axes[0, 0].set_xlabel("PC1")  # Label PC1 coordinate.
axes[0, 0].set_ylabel("PC2")  # Label PC2 coordinate.
axes[0, 0].set_title("Breast Cancer projected to 2 PCs")  # Title the projection panel.
axes[0, 0].legend(*scatter.legend_elements(), title="class")  # Add a class legend.
axes[0, 1].plot(np.arange(1, len(cumulative_cancer) + 1), cumulative_cancer, marker="o", color="black")  # Plot cumulative variance explained.
axes[0, 1].axhline(0.9, color="crimson", linestyle="--", label="90%")  # Mark the 90-percent threshold.
axes[0, 1].set_xlabel("number of components")  # Label component count.
axes[0, 1].set_ylabel("cumulative variance")  # Label cumulative variance fraction.
axes[0, 1].set_ylim(0, 1.03)  # Keep the full variance range visible.
axes[0, 1].set_title("How many PCs retain variance?")  # Title the variance panel.
axes[0, 1].legend()  # Show the threshold label.
axes[1, 0].barh(feature_names_cancer[top_pc1][::-1], pc1_loadings[top_pc1][::-1], color="gray")  # Plot top PC1 loadings.
axes[1, 0].set_title("Largest PC1 loadings")  # Title the PC1 loading panel.
axes[1, 1].barh(feature_names_cancer[top_pc2][::-1], pc2_loadings[top_pc2][::-1], color="royalblue")  # Plot top PC2 loadings.
axes[1, 1].set_title("Largest PC2 loadings")  # Title the PC2 loading panel.
plt.tight_layout()  # Improve dashboard spacing.
plt.show()  # Render projection, variance, and loading interpretation.
print(f"Two-PC cumulative variance: {cumulative_cancer[1]:.3f}")  # Print exact two-component variance retained.
print(f"Two-PC reconstruction MSE in standardized space: {reconstruction_mse_2d:.3f}")  # Print exact reconstruction error.
print("Limitation: overlapping points remain ambiguous because PCA does not use labels.")  # State the unsupervised limitation explicitly.

▶ What you'll see: two PCs reveal broad class structure, but the cumulative-variance curve shows that more components are needed for high-fidelity reconstruction.

👀 **Takeaway:** a complete PCA analysis should include projection, variance retained, reconstruction quality, and loading interpretation—not just a pretty 2-D scatterplot.

---

### Interactive Experiment

Use the slider to choose the number of PCA components for digit reconstruction. Watch the cumulative variance, reconstruction error, and visual quality change together.

In [ ]:
X_interactive = load_digits().data / 16.0  # Load scaled digit pixels for the interactive reconstruction experiment.
example_digit_index = 13  # Choose one digit to visualize while the slider changes.

def show_digit_pca(n_components=8):  # Define the plotting function controlled by the widget slider.
    Z_i, X_hat_i, _, _, ratios_i, _ = pca_from_scratch(X_interactive, k=n_components)  # Fit PCA and reconstruct from the selected number of components.
    cumulative_i = np.cumsum(ratios_i)[n_components - 1]  # Read cumulative variance at the selected component count.
    mse_i = mean_squared_error(X_interactive, X_hat_i)  # Compute reconstruction error for the selected component count.
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))  # Create panels for original, reconstruction, and variance curve.
    axes[0].imshow(X_interactive[example_digit_index].reshape(8, 8), cmap="gray_r")  # Show the original digit image.
    axes[0].set_title("original")  # Title the original panel.
    axes[1].imshow(np.clip(X_hat_i[example_digit_index], 0, 1).reshape(8, 8), cmap="gray_r")  # Show the reconstructed digit image.
    axes[1].set_title(f"reconstruction k={n_components}")  # Title the reconstruction with chosen k.
    axes[2].plot(np.arange(1, len(ratios_i) + 1), np.cumsum(ratios_i), color="black")  # Plot cumulative variance for all components.
    axes[2].axvline(n_components, color="crimson", linestyle="--")  # Mark the slider-selected component count.
    axes[2].set_ylim(0, 1.02)  # Keep variance axis in the 0-to-1 range.
    axes[2].set_xlabel("components")  # Label the component axis.
    axes[2].set_ylabel("cumulative variance")  # Label the variance axis.
    axes[2].set_title(f"variance={cumulative_i:.1%}, MSE={mse_i:.4f}")  # Report variance and error in the title.
    for ax in axes[:2]:  # Loop over image panels.
        ax.set_xticks([])  # Hide image x ticks.
        ax.set_yticks([])  # Hide image y ticks.
    plt.tight_layout()  # Improve spacing.
    plt.show()  # Render the interactive output.

if interact is not None:  # Check whether ipywidgets is available in the current environment.
    interact(show_digit_pca, n_components=IntSlider(value=8, min=1, max=30, step=1, description="components"))  # Create an interactive slider for n_components.
else:  # Handle plain Python environments without widget support.
    show_digit_pca(n_components=8)  # Show a default static version so the code still runs.
    print("Install ipywidgets in a notebook environment for the live slider.")  # Explain how to enable interactivity.

▶ What you'll see: moving the slider right increases variance explained and usually sharpens the reconstruction, but with diminishing returns.

👀 **Takeaway:** choosing $k$ is a tradeoff among compression, reconstruction error, visualization simplicity, and downstream task needs.